# JOUR 701 · Block 3 | One-Way ANOVA
### Same Message Trust dataset · Step-by-step Google Colab lab

**Today's research question:** Does average **message trust** differ across participants with different **education levels**?

- **X / grouping variable:** `education_level` (three categories)
- **Y / outcome:** `message_trust` (numeric)
- **What we will read:** sample size (`n`), group **means**, group **standard deviations (SD)**, the ANOVA **F** and **p-value**, and optional **Tukey pairwise comparisons**.

**Learning goal:** Understand what the code tells us. You do **not** need to memorize Python syntax.

**To open:** Go to [Google Colab](https://colab.research.google.com/) → **File → Upload notebook** → choose this `.ipynb` file. Run from top to bottom using the ▶ icon beside each cell.

## 1 · Quick memory check: which statistical tool?

| Research question | Variable types | Test | Main things to read |
|---|---|---|---|
| Do two numeric measures move together? | numeric + numeric | Pearson correlation | **r**, p |
| Do two groups have different averages? | 2-category group + numeric outcome | Independent-samples t-test | **means, SDs**, p; t for reporting |
| How does expected Y change as X changes? | numeric outcome + predictor | Linear regression | **coefficient b**, intercept, SE / CI, p, R² |
| Do three or more groups have different averages? | 3+-category group + numeric outcome | **One-way ANOVA** | **group means, SDs, F, p**; post-hoc comparisons |

**p-value, in plain English:** *If there truly were no population difference or relationship, how surprising would a result at least this extreme be?* The usual rule is **p < .05 → statistically significant evidence**. A p-value does **not** mean a result is important, and it is **not** the probability that the null hypothesis is true.

In the study, the randomized `condition` has only **two** categories (Control / Personalized), making it a natural t-test example. Here we compare **three education categories**, so we use ANOVA. Education was **not randomly assigned**, so group differences do **not** establish that education caused trust.

## 2 · Load the SAME dataset as the regression class

This is the **same GitHub CSV URL** from the previous correlation / t-test / regression slides. Only the research question changes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

url = ("https://raw.githubusercontent.com/KayChansiri/"
       "Course_research-methods-in-the-age-of-ai/main/data/"
       "JOUR701_Block3_Course_Dataset.csv")

raw = pd.read_csv(url)
print("Rows in source CSV:", len(raw))
print("Repeated participant IDs:", raw.duplicated("participant_id").sum())
display(raw.head())

## 3 · Prepare the two variables we need

Just like last week, **one row = one participant**. The source file contains a repeated participant ID, so we keep one row per participant. We do **not** invent a trust score for a missing response; the ANOVA uses participants with observed values for both variables.

In [ ]:
df = raw.drop_duplicates(subset="participant_id", keep="first").copy()

df["education_level"] = df["education_level"].astype("string").str.strip()
df["message_trust"] = pd.to_numeric(df["message_trust"], errors="coerce")

anova_df = df[["education_level", "message_trust"]].dropna().copy()
anova_df["education_level"] = anova_df["education_level"].astype(str)

print("Unique participants after removing duplicate:", len(df))
print("Participants with both ANOVA variables:", len(anova_df))
print("Education groups:")
print(anova_df["education_level"].value_counts())

**Checkpoint:** What makes `education_level` categorical? What makes `message_trust` numeric? What does a missing value mean for this analysis?

## 4 · Look at each group's mean and SD **before** the test

- **Mean:** average trust score in a group
- **SD:** how spread out the trust scores are inside that group
- **n:** how many participants had nonmissing scores in that group

These are **descriptive statistics**. They tell you what the sample looks like, even before you know the p-value.

In [ ]:
education_order = [
    "High school / some college",
    "Bachelor’s degree",
    "Graduate degree",
]

summary = (anova_df.groupby("education_level")["message_trust"]
           .agg(n="count", mean="mean", SD="std")
           .reindex(education_order))

display(summary.round(2))
print("Highest sample mean:", summary["mean"].idxmax())
print("Lowest sample mean:", summary["mean"].idxmin())

**Write one sentence:** “In this sample, the _____ group had the highest average message trust.”

Do **not** call a difference statistically significant yet. We have not run a test.

## 5 · Picture the three groups

**Dots = individual participants. Short dark horizontal lines = group averages.** Look for the difference **between** group centers and the spread **within** each group.

In [ ]:
short_labels = ["HS / some college", "Bachelor's", "Graduate"]
rng = np.random.default_rng(701)  # Same dot positions each time
fig, ax = plt.subplots(figsize=(9, 5))

for i, group_name in enumerate(education_order):
    y = anova_df.loc[anova_df["education_level"] == group_name,
                     "message_trust"].to_numpy()
    x = rng.normal(loc=i, scale=0.07, size=len(y))
    ax.scatter(x, y, s=24, alpha=0.45, color="#2E70B5")
    ax.hlines(y.mean(), i - 0.22, i + 0.22,
              colors="#174B76", linewidth=4)

ax.set_xticks(range(3), short_labels)
ax.set_ylabel("Message trust (numeric score)")
ax.set_xlabel("Education level (three groups)")
ax.set_title("Message Trust by Education Level")
ax.set_xlim(-0.5, 2.5)
ax.grid(axis="y", alpha=0.18)
plt.tight_layout()
plt.show()

**Notice:** Even if two means look different, many individual scores may overlap. ANOVA compares **between-group differences** to **within-group spread**.

## 6 · Run the one-way ANOVA

- **H₀ (null):** all three **population means** are equal.
- **H₁ (alternative):** **at least one** population mean is different.

Read the formula as English: `message_trust ~ C(education_level)` = “model message trust using education **categories**.”

Read **F** and **PR(>F)** from the `C(education_level)` row. **PR(>F)** is the p-value.

In [ ]:
model = smf.ols("message_trust ~ C(education_level)", data=anova_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

display(anova_table.round(4))

F_value = anova_table.loc["C(education_level)", "F"]
p_value = anova_table.loc["C(education_level)", "PR(>F)"]
ss_between = anova_table.loc["C(education_level)", "sum_sq"]
ss_within = anova_table.loc["Residual", "sum_sq"]
eta_squared = ss_between / (ss_between + ss_within)

p_print = f"{p_value:.4f}" if p_value >= 0.0001 else "< .0001"
print(f"F = {F_value:.3f}")
print(f"p = {p_print}")
print(f"Eta-squared (share of sample variation associated with group) = {eta_squared:.3f}")
if p_value < 0.05:
    print("Decision: p < .05; evidence that at least one population group mean differs.")
else:
    print("Decision: p ≥ .05; not enough evidence to conclude the population means differ.")

### What does that actually mean?

**Think of ANOVA as asking ONE question first:** “Is there evidence that the group averages are *not all the same*?”

- **F:** difference between group means **relative to** ordinary differences among people within groups. Bigger F can be stronger evidence, but **read p** to make the decision.
- **p < .05:** statistically significant evidence that **at least one** group mean differs. It does **not** identify which two groups differ.
- **p ≥ .05:** not enough evidence of a difference; it does **not** prove all means are identical.
- **Eta-squared:** estimated proportion of **observed sample variation** associated with education group in this one-way analysis. It is **not** percentage accuracy.

**Do not say:** “The p-value is the chance the null hypothesis is true,” “the result must be p < .05,” or “education causes changes in trust.”

## 7 · Which groups differ? (Tukey follow-up)

The overall ANOVA tells us whether **at least one** mean differs. **Tukey HSD** compares each **pair** of groups and adjusts for multiple comparisons.

Read **`group1`, `group2`, `meandiff`, `p-adj`, `reject`**. In this output, **`reject = True`** means that pair differs at the .05 familywise level. We display the comparisons for practice even if the overall ANOVA is not significant; do **not** search for a positive finding.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=anova_df["message_trust"],
    groups=anova_df["education_level"],
    alpha=0.05,
)
print(tukey.summary())

if p_value >= 0.05:
    print("Reminder: overall ANOVA was not significant; avoid claiming group differences.")

## 8 · One-sentence interpretation (fill in YOUR numbers)

**ANOVA template**

> We compared average **message trust** across **three education groups** using one-way ANOVA. The mean trust scores were **[insert group means and SDs]**. The overall ANOVA was **[statistically significant / not statistically significant]**, **F**([df between], [df within]) = **[F]**, **p** = **[p]**. **[If significant, name any specific pairs confirmed by Tukey; otherwise say there was insufficient evidence of a group difference.]**

**Important:** Education level is a participant characteristic, **not a randomized assignment**. Here we can describe **differences / associations**, not causal effects.

### Your turn: write THREE short answers

1. Which group had the highest mean? What is its **SD**?
2. Was the overall ANOVA statistically significant using the conventional **.05** cutoff?
3. Can this analysis show that more education **causes** higher trust? Why or why not?

## 9 · (Optional) Try a DIFFERENT three-or-more-group variable

Try `primary_platform` (Instagram, TikTok, YouTube, X) instead of `education_level`. The research question becomes: “Does average message trust differ across participants whose **primary social media platforms** are different?”

⚠️ Platform is **not randomly assigned** either. **Do not claim a causal effect.** Clean the lowercase variant `instagram` first.

In [ ]:
platform_df = df[["primary_platform", "message_trust"]].dropna().copy()
platform_df["primary_platform"] = (platform_df["primary_platform"]
                                    .astype(str).str.strip()
                                    .replace({"instagram": "Instagram"}))

display(platform_df.groupby("primary_platform")["message_trust"]
        .agg(n="count", mean="mean", SD="std").round(2))

platform_model = smf.ols(
    "message_trust ~ C(primary_platform)", data=platform_df
).fit()
display(sm.stats.anova_lm(platform_model, typ=2).round(4))

## 10 · Bonus (only if you have time): a simple assumption check

The ordinary one-way ANOVA works best when each row represents an independent person, the data are not dominated by extreme values, and **group spreads** are reasonably comparable. Larger group sizes help make ANOVA more robust to modest deviations from normality. The next cell tests whether the groups have **similar spreads** using the median-centered Levene test; this check is only one piece of model assessment.

**Levene p < .05** suggests different variances (unlike the ANOVA p-value, it is **not** evidence of different group means). If spreads are very unequal, a **Welch ANOVA** is often preferable; ask your instructor before choosing a different analysis.

In [ ]:
from scipy.stats import levene

groups = [anova_df.loc[anova_df["education_level"] == level,
                       "message_trust"].to_numpy()
          for level in education_order]

lev_stat, lev_p = levene(*groups, center="median")
print(f"Levene's test for similar spreads: p = {lev_p:.4f}")
print("This checks spread, NOT whether group means differ.")

---

### Final memory cue

**Two numeric variables → correlation.**  **Two groups → t-test.**  **Three or more groups → ANOVA.**  **Predict a numeric outcome → regression.**

In reality, the **research question**, **variable roles**, **design**, and **assumptions** matter too. A two-group ANOVA is mathematically equivalent to a pooled-variance two-sample t-test (**F = t²** when the same assumptions/model are used). One-way ANOVA can be fit using regression with a categorical predictor: they are connected members of the general linear model family.

**Next step:** Complete the three short answers in Section 8 and bring your interpretation to class.